# 책장 넘기는 소리만 추출하기

오디오북 MP3(Disney Fun to Read Set 1, 23트랙)에서 **책장 넘기는 소리만** 자동 검출·절단하는 노트북입니다.

## 소리 특성 (실측 보정값)

| 구분 | RMS 레벨 | 스펙트럼 중심주파수 | 길이 |
|---|---|---|---|
| 음성 | -9 ~ -15 dB | 400 ~ 1,300 Hz | 다양 |
| **책장 소리** | **-17 ~ -19 dB** (조용한 케이스는 -24 ~ -44 dB) | **4,100 ~ 12,000 Hz** | 0.3 ~ 1.0 s |
| 무음 | -60 dB 이하 (디지털 무음) | - | - |

## 파이프라인
1. **[1단계] 후보 검출** — 프레임별 RMS + 스펙트럼 중심주파수에서 `centroid > 4kHz` 이고 `crest factor > 19dB` 인 구간을 책장 소리 후보로 선정
2. **[2단계] 버스트 절단** — 후보 중심 ±5s 창을 무음(-60dB, 0.15s 이상) 기준으로 버스트 분리 → 레벨·길이·스펙트럼으로 책장 소리 버스트만 선택 → 여백(앞 0.15s / 뒤 0.25s)을 두되 인접 음성을 침범하지 않도록 절단
3. **[자동 검증]** 클립 안에 음성(-15.5dB 초과)이 남아 있는지 확인
4. **[수동 검증]** 클립을 직접 들어보고 오탐 제거 (6번 셀)

필요 패키지: `pip install soundfile numpy scipy`

In [ ]:
import os, csv, glob
import numpy as np
import soundfile as sf
from scipy import signal

# ─────────────────────────────────────────────
# 경로 설정 (다른 데이터셋에 적용할 때 여기만 수정)
# ─────────────────────────────────────────────
MP3_FOLDER = 'D:/SEMCOWork/Session26_mp3/Disney Fun to Read 1'      # 원본 MP3 폴더
OUT_DIR    = 'D:/SEMCOWork/Session26_mp3/page_turn_only'             # 클립 저장 폴더
IDX_CSV    = 'D:/SEMCOWork/Session26_mp3/page_turn_only_index.csv'   # 결과 인덱스 CSV
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# 파라미터 — 23개 트랙 · 348개 이벤트로 실측 보정한 값
#   음성    : RMS -9 ~ -15 dB, 스펙트럼 중심 400 ~ 1,300 Hz
#   책장소리: RMS -17 ~ -19 dB(조용한 케이스 -24 ~ -44), 중심 4,100 ~ 12,000 Hz, 0.3 ~ 1.0 s
# ─────────────────────────────────────────────
CFG = dict(
    # [1단계] 후보 검출
    frame_rms_t=-42.0,     # 프레임 활성 판정 레벨(dB)
    centroid_t=4000.0,     # 스펙트럼 중심주파수 하한(Hz) — 음성(~1kHz)과 분리
    crest_t=19.0,          # crest factor 하한(dB) — 책장소리 ~22.6, 음성 9 ~ 17
    run_min=0.3,           # 후보 구간 최소 길이(s)
    run_max=1.6,           # 후보 구간 최대 길이(s)
    merge_gap=0.25,        # 프레임 병합 간격(s)
    # [2단계] 버스트 분리
    sil_t=-60.0,           # 무음 판정 레벨(dB)
    min_sil=0.15,          # 무음 최소 지속(s)
    # 책장소리 버스트 선택(1차)
    pt_rms=(-22.0, -15.5), # 일반 책장소리 레벨 범위(dB)
    pt_dur=(0.3, 2.0),     # 버스트 길이 범위(s)
    pt_cen=3500.0,         # centroid 최댓값 하한(Hz)
    # 폴백(조용하거나 음성에 붙은 케이스)
    fb_rms_max=-17.0,
    fb_cen_med=2500.0,
    # 절단 / 검증
    pad_before=0.15,       # 앞 여백(s) — 인접 버스트 침범 시 자동 클램프
    pad_after=0.25,        # 뒤 여백(s)
    voice_t=-15.5,         # 음성 판정 레벨(dB) — 검증용
)
print('설정 완료:', MP3_FOLDER)

## 1. 공통 함수
오디오 로드, 프레임 RMS, 스펙트럼 중심주파수, 무음 기반 버스트 분리.

In [ ]:
def load_mono(path):
    '''MP3/WAV -> 모노 float64 배열, 샘플레이트'''
    data, sr = sf.read(path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    return data.astype(np.float64), sr


def frame_rms_db(x, sr, win=0.02):
    '''20ms 창 / 10ms hop 프레임 RMS(dB). 반환: (시간[프레임 중심], dB'''
    n = int(sr * win)
    h = n // 2
    if len(x) < n:
        return np.array([]), np.array([])
    frames = np.lib.stride_tricks.sliding_window_view(x, n)[::h]
    rms = np.sqrt(np.mean(frames ** 2, axis=1))
    t = (np.arange(len(frames)) * h + n / 2) / sr
    return t, 20 * np.log10(rms + 1e-12)


def stft_centroid(x, sr, n_fft=2048, hop=512):
    '''STFT 프레임별 스펙트럼 중심주파수(Hz). 반환: (시간, centroid'''
    f, t, Z = signal.stft(x, fs=sr, nperseg=n_fft, noverlap=n_fft - hop,
                          padded=False, boundary=None)
    mag = np.abs(Z) + 1e-12
    return t, (f[:, None] * mag).sum(axis=0) / mag.sum(axis=0)


def segment_bursts(x, sr, sil_t, min_sil, win=0.02):
    '''무음(sil_t 이하가 min_sil 이상 지속)을 경계로 버스트 분리. 반환: [(t0, t1), ...]'''
    t, db = frame_rms_db(x, sr, win)
    if len(t) == 0:
        return []
    hop = win / 2
    idx = np.where(db > sil_t)[0]
    if len(idx) == 0:
        return []
    splits = np.where(np.diff(idx) * hop >= min_sil)[0]
    return [(t[g[0]] - hop / 2, t[g[-1]] + hop / 2)
            for g in np.split(idx, splits + 1)]

## 2. [1단계] 후보 검출
책장 소리는 음성보다 스펙트럼 중심주파수가 높고(> 4kHz), 순간 피크가 커서(crest factor > 19dB) 이 조건으로 후보를 좁힙니다.

In [ ]:
def detect_candidates(path, cfg):
    '''[1단계] 책장 소리 후보 구간 검출.
    centroid > centroid_t 이고 프레임 RMS > frame_rms_t 인 프레임을
    merge_gap 이하 간격으로 병합한 뒤, 길이/crest/centroid 조건으로 필터.
    반환: (후보 리스트, sr, 오디오 배열)'''
    x, sr = load_mono(path)
    t, db = frame_rms_db(x, sr)
    tc, cen = stft_centroid(x, sr)
    cen_r = np.interp(t, tc, cen)
    idx = np.where((cen_r > cfg['centroid_t']) & (db > cfg['frame_rms_t']))[0]
    if len(idx) == 0:
        return [], sr, x
    hop = 0.01
    splits = np.where(np.diff(idx) * hop > cfg['merge_gap'])[0]
    cands = []
    for g in np.split(idx, splits + 1):
        t0, t1 = t[g[0]], t[g[-1]]
        if not (cfg['run_min'] <= t1 - t0 <= cfg['run_max']):
            continue
        seg = x[int(t0 * sr):int(t1 * sr)]
        rms = np.sqrt(np.mean(seg ** 2) + 1e-12)
        crest = 20 * np.log10((np.max(np.abs(seg)) + 1e-12) / rms)
        if crest < cfg['crest_t'] or cen_r[g].mean() < cfg['centroid_t']:
            continue
        cands.append(dict(center=(t0 + t1) / 2, t0=t0, t1=t1, crest=crest,
                          cen=float(cen_r[g].mean()), rms=20 * np.log10(rms)))
    return cands, sr, x


## 3. [2단계] 책장 소리 버스트 절단
후보 중심 ±5s 창을 무음 기준으로 버스트로 분리한 뒤, 레벨(-22 ~ -15.5dB)·길이(0.3 ~ 2s)·스펙트럼(centroid > 3.5kHz)으로 책장 소리 버스트만 고릅니다. 조용하거나 음성에 붙은 케이스는 폴백 조건(centroid 중앙값 > 2.5kHz)으로 처리합니다.

In [ ]:
def extract_pt_clip(x, sr, cand, cfg):
    '''[2단계] 책장 소리 버스트만 절단.
    선택: 1차 = 레벨(pt_rms) + 길이(pt_dur) + 스펙트럼(cen_max > pt_cen)
          폴백 = 조용한 버스트(maxr < fb_rms_max, cen_med > fb_cen_med)
    반환: (결과 dict 또는 None, 버스트 특징 리스트)'''
    c = cand['center']
    w0 = max(0.0, c - 5.0)
    w1 = min(len(x) / sr, c + 5.0)
    seg = x[int(w0 * sr):int(w1 * sr)]
    feats = []
    for b0, b1 in segment_bursts(seg, sr, cfg['sil_t'], cfg['min_sil']):
        s = seg[int(b0 * sr):int(b1 * sr)]
        if len(s) < sr * 0.05:
            continue
        _, cen = stft_centroid(s, sr)
        _, fdb = frame_rms_db(s, sr)
        feats.append(dict(t0=w0 + b0, t1=w0 + b1, dur=b1 - b0,
                          maxr=float(fdb.max()) if len(fdb) else -240.0,
                          cen_max=float(cen.max()), cen_med=float(np.median(cen))))
    prim = [f for f in feats
            if cfg['pt_rms'][0] <= f['maxr'] <= cfg['pt_rms'][1]
            and cfg['pt_dur'][0] <= f['dur'] <= cfg['pt_dur'][1]
            and f['cen_max'] > cfg['pt_cen']]
    fb = [f for f in feats
          if f['maxr'] < cfg['fb_rms_max'] and f['cen_med'] > cfg['fb_cen_med']
          and cfg['pt_dur'][0] <= f['dur'] <= cfg['pt_dur'][1]]
    pool = prim if prim else fb
    if not pool:
        return None, feats
    # 후보 구간과 겹치는 버스트를 우선 선택(창 안의 인접 페이지 넘김 오선택 방지),
    # 겹치는 것이 없으면 후보 중심에 가장 가까운 버스트 사용
    ov = [f for f in pool if f['t0'] < cand['t1'] and f['t1'] > cand['t0']]
    base = ov if ov else pool
    best = min(base, key=lambda f: abs((f['t0'] + f['t1']) / 2 - c))
    # 여백이 인접 버스트(음성)를 침범하지 않도록 클램프
    prev_end = max([f['t1'] for f in feats if f['t1'] <= best['t0'] + 1e-9], default=w0)
    next_beg = min([f['t0'] for f in feats if f['t0'] >= best['t1'] - 1e-9], default=w1)
    cs = max(best['t0'] - cfg['pad_before'], prev_end + 0.02, w0)
    ce = min(best['t1'] + cfg['pad_after'], next_beg - 0.02, w1)
    if ce - cs < 0.2:
        return None, feats
    clip = x[int(cs * sr):int(ce * sr)]
    return dict(clip=clip, cs=cs, ce=ce, pt=(best['t0'], best['t1']), best=best), feats

## 4. 전체 실행 — 클립 저장 + 인덱스 CSV
폴더의 모든 MP3를 처리해 `page_turn_only/` 에 WAV 클립을 저장하고 인덱스를 기록합니다.

In [ ]:
mp3s = sorted(glob.glob(os.path.join(MP3_FOLDER, '*.mp3')))
print(f'대상 파일: {len(mp3s)}개')
print()

rows, fails = [], []
for i, path in enumerate(mp3s, 1):
    cands, sr, x = detect_candidates(path, CFG)
    n_ok = 0
    for j, cand in enumerate(cands, 1):
        c = cand['center']
        res, feats = extract_pt_clip(x, sr, cand, CFG)
        if res is None:
            fails.append((i, j, round(c, 2), '버스트 선택 실패'))
            continue
        clip = res['clip']
        p0, p1 = res['pt']
        b = res['best']
        bmaxr = b['maxr']
        bcen = b['cen_med']
        # 자동 검증: 책장 소리 구간 밖에 음성(> voice_t)이 남으면 실패
        tc, db = frame_rms_db(clip, sr)
        pt_m = (tc >= p0 - res['cs'] - 0.05) & (tc <= p1 - res['cs'] + 0.05)
        if (~pt_m).any():
            leak = bool((db[~pt_m] > CFG['voice_t']).any())
        else:
            leak = False
        fname = f'track{i:02d}_pt{j:02d}_{c:.2f}s.wav'
        sf.write(os.path.join(OUT_DIR, fname), clip, sr)
        rows.append(dict(track=i, file=os.path.basename(path),
                         src_start=f'{c:.2f}',
                         pt_start=f'{p0:.2f}', pt_end=f'{p1:.2f}',
                         pt_dur=f'{p1 - p0:.2f}',
                         clip_dur=f'{len(clip) / sr:.3f}',
                         maxr=f'{bmaxr:.2f}', cen_med=f'{bcen:.0f}',
                         verified=str(not leak), clip=fname))
        if leak:
            fails.append((i, j, round(c, 2), 'voice_leak(검증 실패)'))
        else:
            n_ok += 1
    print(f'  track {i:2d} | 후보 {len(cands):2d}개 | 저장 {n_ok:2d}개')

with open(IDX_CSV, 'w', newline='', encoding='utf-8-sig') as fp:
    w = csv.DictWriter(fp, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print()
print(f'저장: {len(rows)}개 -> {OUT_DIR}')
print(f'인덱스: {IDX_CSV}')
if fails:
    print()
    print('[실패/주의] (track, pt, 시간, 사유)')
    for item in fails:
        print('  ', item)

## 5. 결과 검증
클립 길이 분포와 트랙별 첫 책장 소리 시간을 확인합니다. 정상이면 모든 트랙의 첫 책장 소리가 20.8 ~ 21.3s 에 나타납니다.

In [ ]:
durs = [float(r['clip_dur']) for r in rows]
print(f'클립 수: {len(rows)}개')
print(f'길이: min={min(durs):.2f}s / median={np.median(durs):.2f}s / max={max(durs):.2f}s')
print(f'총 재생시간: {sum(durs) / 60:.1f}분')
n_leak = sum(1 for r in rows if r['verified'] == 'False')
print(f'자동 검증 실패(voice leak): {n_leak}개')
print()
print('트랙별 첫 책장 소리 시간 (정상이면 전 트랙 ~20.8 ~ 21.3s):')
first = {}
for r in rows:
    if r['track'] not in first:
        first[r['track']] = float(r['pt_start'])
line = ', '.join(f'{k}:{v:.1f}s' for k, v in sorted(first.items()))
print('  ' + line)

## 6. 수동 검증 — 반드시 샘플링해서 들어보기

자동 검출은 대부분 정확하지만, 음성·효과음과 비슷한 소리는 오탐일 수 있습니다.
(실제 검증에서 `track01_pt04_51.16s.wav` 는 책장 소리가 아닌 것으로 확인되어 제거했습니다.)

`play()` 로 들어보고 책장 소리가 아니면 `remove_clip()` 으로 제거하세요.

In [ ]:
from IPython.display import Audio, display

def play(clip_name):
    '''클립을 노트북에서 바로 재생해서 확인'''
    display(Audio(os.path.join(OUT_DIR, clip_name)))

def remove_clip(clip_name):
    '''들어본 결과 오탐이면 제거 + 인덱스 갱신'''
    p = os.path.join(OUT_DIR, clip_name)
    if os.path.exists(p):
        os.remove(p)
    with open(IDX_CSV, 'r', encoding='utf-8-sig') as fp:
        rws = list(csv.DictReader(fp))
    rws = [r for r in rws if r['clip'] != clip_name]
    with open(IDX_CSV, 'w', newline='', encoding='utf-8-sig') as fp:
        w = csv.DictWriter(fp, fieldnames=list(rws[0].keys()))
        w.writeheader()
        w.writerows(rws)
    print(f'제거: {clip_name} | 인덱스 {len(rws)}행')

# 사용 예:
# play('track01_pt01_20.84s.wav')
# remove_clip('track01_pt04_51.16s.wav')   # 실제 검증에서 오탐으로 확인된 클립

## 튜닝 가이드 (다른 오디오북에 적용할 때)

| 증상 | 조정 |
|---|---|
| 책장 소리가 검출 안 됨 | `crest_t` ↓ (19 → 17), `centroid_t` ↓ (4000 → 3500) |
| 음성이 클립에 섞임 | `voice_t` ↓, `pad_before`/`pad_after` ↓, `pt_rms` 상한 ↓ |
| 클립이 너무 짧게 잘림 | `min_sil` ↑ (0.15 → 0.25) — 무음 판정을 관대하게 |
| 오탐이 많음 | `pt_rms` 범위를 실측 레벨에 맞게 좁힘 |

### 인덱스 CSV 컬럼
- `track` 트랙 번호 / `file` 원본 파일명
- `src_start` 검출 후보 중심(원본 시간, s)
- `pt_start`/`pt_end` 책장 소리 버스트 시작·끝(원본 시간, s) / `pt_dur` 버스트 길이
- `clip_dur` 저장 클립 길이(s) / `maxr` 버스트 최대 레벨(dB) / `cen_med` 중심주파수 중앙값(Hz)
- `verified` 자동 검증 통과 여부 / `clip` 클립 파일명